# Predicción de salarios

Continuamos con el ejercicio de predicción de salarios, añadiendo los estimadores de Lasso y Post-Lasso. Veremos como ellos resuelven el problema de dimensionalidad

In [ ]:
suppressWarnings(
pacman::p_load(xtable, dplyr, hdm, glmnet)
)

In [ ]:
file<-"https://raw.githubusercontent.com/andvarga-eco/econometrics_ml/refs/heads/main/data/geih_24_baq.csv"
data<-read.csv(file)

In [ ]:
data<-data|>filter(P6040 %in% 25:64)|>filter(P3042!=99)
data<-na.omit(data)
data<-data|>filter(P6800<=60)|>mutate(w=INGLABO/(P6800*4),lw=log(w),
                                      edad2=P6040^2, edad3=P6040^3, edad4=P6040^4)

## Partir la muestra

In [ ]:
# Partir los datos en data de entrenamiento (80%) y data de testeo (20%)
set.seed(1) # para poder replicar
n <- nrow(data)
random <- sample(1:n, floor(n * 4 / 5)) # draw (4/5)*n random numbers de 1 a n sin reemplazo
train <- data[random, ]
test <- data[-random, ]

### Predicción dentro de muestra

In [ ]:
# Modelos
basic<-lw~(C(P3271)+P6040+C(P3042)+C(RAMA2D_R4))
flex<-lw~C(P3271)+C(P3042)+C(RAMA2D_R4)+(P6040+edad2+edad3+edad4)*(C(P3271)+C(P3042)+C(RAMA2D_R4))

In [ ]:
# 1. Modelo básico
regbasic <- lm(basic, data = train)
# número de regresores
cat("Number of regressors in the basic model:", length(regbasic$coef), "\n")

In [ ]:
# 2. Modelo flexible
regflex <- lm(flex, data = train)
# número de regresores
cat("Number of regressors in the flex model:", length(regflex$coef), "\n")

In [ ]:
# 3. Flexible usando Lasso
train_flex <- model.matrix(flex, train) # all regressors
fit_lasso_cv <- cv.glmnet(as.matrix(train_flex), train$lw, family = "gaussian", alpha = 1, nfolds = 5)
yhat_lasso_cv <- predict(fit_lasso_cv, newx = as.matrix(train_flex), s = "lambda.min")

In [ ]:
print(fit_lasso_cv)

In [ ]:
coef_lasso<-coef(fit_lasso_cv,s="lambda.min")
nz <- sum(coef_lasso[-1] != 0)                 # excluye el intercepto
cat("Número de coeficientes distintos de cero con lambda.min:", nz, "\n")

Estadísticas de predicción dentro de muestra

In [ ]:
sumbasic <- summary(regbasic)
sumflex <- summary(regflex)

ntrain <- nrow(train)

# R-squared and adjusted R-squared
r2_1 <- sumbasic$r.squared
cat("R-squared for the basic model: ", r2_1, "\n")
r2_adj1 <- sumbasic$adj.r.squared
cat("adjusted R-squared for the basic model: ", r2_adj1, "\n")

r2_2 <- sumflex$r.squared
cat("R-squared for the flexible model: ", r2_2, "\n")
r2_adj2 <- sumflex$adj.r.squared
cat("adjusted R-squared for the flexible model: ", r2_adj2, "\n")

p_l <- fit_lasso_cv$nzero[fit_lasso_cv$index[1]]
r2_l <- 1 - sum((yhat_lasso_cv - train$lw)^2) / sum((train$lw - mean(train$lw))^2)
cat("R-squared for the lasso with flexible model: ", r2_l, "\n")
r2_adj_l <- 1 -
  (sum((yhat_lasso_cv - train$lw)^2) / (ntrain - p_l - 1)) /
    (sum((train$lw - mean(train$lw))^2) / (ntrain - 1))
cat("adjusted R-squared for the flexible model: ", r2_adj_l, "\n")


# MSE
mse1 <- mean(sumbasic$res^2)
cat("MSE for the basic model: ", mse1, "\n")
p1 <- sumbasic$df[1] # number of regressors
mse_adj1 <- (ntrain / (ntrain - p1)) * mse1
cat("adjusted MSE for the basic model: ", mse_adj1, "\n")

mse2 <- mean(sumflex$res^2)
cat("MSE for the flexible model: ", mse2, "\n")
p2 <- sumflex$df[1]
mse_adj2 <- (ntrain / (ntrain - p2)) * mse2
cat("adjusted MSE for the lasso flexible model: ", mse_adj2, "\n")

lasso_res <- train$lw - yhat_lasso_cv
mse_l <- mean(lasso_res^2)
cat("MSE for the lasso flexible model: ", mse_l, "\n")
mse_adj_l <- (ntrain / (ntrain - p_l - 1)) * mse_l
cat("adjusted MSE for the lasso flexible model: ", mse_adj_l, "\n")



In [ ]:
table <- matrix(0, 3, 5)
table[1, 1:5] <- c(p1, r2_1, mse1, r2_adj1, mse_adj1)
table[2, 1:5] <- c(p2, r2_2, mse2, r2_adj2, mse_adj2)
table[3, 1:5] <- c(p_l, r2_l, mse_l, r2_adj_l, mse_adj_l)
colnames(table) <- c("p", "$R^2_{sample}$", "$MSE_{sample}$", "$R^2_{adjusted}$", "$MSE_{adjusted}$")
rownames(table) <- c("basic reg", "flexible reg", "lasso flex")
tab <- xtable(table, digits = c(0, 0, 2, 2, 2, 2))
tab

## Predicción fuera de muestra

In [ ]:
library(dplyr)

test %>%
  anti_join(distinct(train, RAMA2D_R4), by = "RAMA2D_R4") %>%
  count(RAMA2D_R4, sort = TRUE)

In [ ]:
# 1. Modelo básico
options(warn = -1)
test_ad<-test|>filter(RAMA2D_R4!="30" & RAMA2D_R4!="3")
yhat_bas <- predict(regbasic, newdata = test_ad)
y_test <- test_ad$lw
mean_train <- mean(train$lw)
mse_test1 <- sum((y_test - yhat_bas)^2) / length(y_test)
r2_test1 <- 1 - mse_test1 / mean((y_test - mean_train)^2)

cat("Test MSE for the basic model: ", mse_test1, " ")
cat("Test R2 for the basic model: ", r2_test1)

In [ ]:
# 2. Modelo flexible

options(warn = -1)

yhat_flex <- predict(regflex, newdata = test_ad)
y_test <- test_ad$lw
mean_train <- mean(train$lw)
mse_test2 <- sum((y_test - yhat_flex)^2) / length(y_test)
r2_test2 <- 1 - mse_test2 / mean((y_test - mean_train)^2)

cat("Test MSE for the flexible model: ", mse_test2, " ")

cat("Test R2 for the flexible model: ", r2_test2)

In [ ]:
# 3. Lasso

flex_data <- model.matrix(flex, data)
train_flex <- flex_data[random, ]
test_flex <- flex_data[-random, ]
y_test<-test$lw
fit_lasso_cv <- cv.glmnet(train_flex, train$lw, family = "gaussian", alpha = 1, nfolds = 5)
yhat_lasso_cv <- predict(fit_lasso_cv, newx = test_flex, s = "lambda.min")

# calculating the out-of-sample MSE
mse_lasso <- sum((y_test - yhat_lasso_cv)^2) / length(y_test)
r2_lasso <- 1 - mse_lasso / mean((y_test - mean(train$lw))^2)

cat("Test MSE for the lasso on flexible model: ", mse_lasso, " ")

cat("Test R2 for the lasso flexible model: ", r2_lasso)

In [ ]:
table2 <- matrix(0, 3, 2)
table2[1, 1] <- mse_test1
table2[2, 1] <- mse_test2
table2[3, 1] <- mse_lasso
table2[1, 2] <- r2_test1
table2[2, 2] <- r2_test2
table2[3, 2] <- r2_lasso

rownames(table2) <- c("basic reg", "flexible reg", "lasso regression")
colnames(table2) <- c("$MSE_{test}$", "$R^2_{test}$")
tab2 <- xtable(table2, digits = 3)
tab2

Note que las medidas de desempeño de Lasso son similares en la predicción fuera de muestra con las realizadas dentro de muestra. En contrate, MCO se deteriora bastante cuando p/n no es pequeño, modelo flexible